[![pythonista.io](img/pythonista.png)](https://www.pythonista.io)

# Linters y tipado en pipeline.

Py271 toma la estructura del workflow y la convierte en quality gates reales con ruff, mypy y pytest.

## Objetivos.

- Entender que es un quality gate.
- Ver por que lint y tipado deben bloquear merges.
- Preparar la base del job de verificaciones.

## Del workflow de ejemplo al workflow que bloquea.

En vez de usar Actions solo para imprimir mensajes, Py271 lo usa para decidir si un cambio puede avanzar. Un quality gate es una condicion automatizada que detiene el pipeline cuando el codigo no cumple un umbral minimo.

Para este curso usaremos principalmente:

- `ruff` para reglas estaticas y estilo.
- `mypy` para contratos de tipos.
- `pytest` para comportamiento verificable.

## El ecosistema anterior: por que habia tantas herramientas.

Antes de ruff, un proyecto Python bien configurado requeria varias herramientas independientes para cubrir las distintas dimensiones de calidad:

- **flake8**: verificaba PEP 8, errores de sintaxis e imports no utilizados. Era rapido pero superficial; no detectaba problemas logicos ni de complejidad.
- **pylint**: realizaba analisis mas profundo — complejidad ciclomatica, convencion de nombres, docstrings, dependencias circulares. Era exhaustivo pero notoriamente lento y generaba muchos falsos positivos que los equipos aprendian a ignorar.
- **isort**: ordenaba y agrupaba los imports segun la convencion (stdlib, terceros, locales). Requeria su propio archivo de configuracion y podia entrar en conflicto con black.
- **black**: formateador opinionado que eliminaba los debates de estilo. No analizaba logica; solo reformateaba. Su configuracion era minima por diseno.
- **bandit**: escaneaba el codigo en busca de patrones de seguridad: uso inseguro de `subprocess`, secretos embebidos, llamadas peligrosas. Herramienta independiente con su propia cadena de ejecucion.

El problema de este ecosistema en un pipeline CI/CD era triple: **velocidad** (ejecutar cinco herramientas en secuencia sumaba minutos), **configuracion** (cinco archivos o secciones separadas que podian contradecirse entre si), y **mantenimiento** (versiones y actualizaciones independientes para cada herramienta).

La celda siguiente muestra el mapa completo de herramientas y como ruff consolida la mayoria de ellas.

In [1]:
herramientas = [
    ("flake8",  "linter",       "PEP 8, errores de sintaxis, imports no usados. Rapido pero superficial."),
    ("pylint",  "linter",       "Analisis profundo: complejidad, nombres, docstrings. Lento y verboso."),
    ("isort",   "formato",      "Ordena y agrupa imports segun convencion. Archivo de config separado."),
    ("black",   "formato",      "Formateador sin opciones. Elimina debates de estilo pero no analiza."),
    ("bandit",  "seguridad",    "Detecta patrones inseguros en codigo Python (inyecciones, secretos)."),
    ("mypy",    "tipos",        "Validacion estatica de type hints. No reemplazable por ruff."),
    ("ruff",    "todo lo anterior (menos mypy)",
                                "Reemplaza flake8 + pylint + isort + black + bandit en un solo binario."),
]

print(f"{'Herramienta':<10} {'Rol':<35} Descripcion")
print("-" * 95)
for nombre, rol, descripcion in herramientas:
    print(f"{nombre:<10} {rol:<35} {descripcion}")

Herramienta Rol                                 Descripcion
-----------------------------------------------------------------------------------------------
flake8     linter                              PEP 8, errores de sintaxis, imports no usados. Rapido pero superficial.
pylint     linter                              Analisis profundo: complejidad, nombres, docstrings. Lento y verboso.
isort      formato                             Ordena y agrupa imports segun convencion. Archivo de config separado.
black      formato                             Formateador sin opciones. Elimina debates de estilo pero no analiza.
bandit     seguridad                           Detecta patrones inseguros en codigo Python (inyecciones, secretos).
mypy       tipos                               Validacion estatica de type hints. No reemplazable por ruff.
ruff       todo lo anterior (menos mypy)       Reemplaza flake8 + pylint + isort + black + bandit en un solo binario.


## ruff: linter y formateador unificado.

`ruff` es una herramienta de analisis estatico y formato de codigo Python escrita en Rust. Reemplaza `flake8`, `pylint`, `isort`, `black` y varias de sus extensiones en un solo binario con configuracion unificada.

Las razones para usarlo en un pipeline CI/CD son:

- **velocidad**: analiza proyectos grandes en milisegundos, reduciendo el tiempo total del workflow.
- **unificacion**: un solo archivo de configuracion reemplaza multiples herramientas con configuraciones potencialmente contradictorias.
- **reglas de seguridad**: incluye reglas del grupo `S` (derivadas de `bandit`) que detectan patrones de codigo inseguro como uso de `subprocess` sin sanitizar o secretos embebidos.
- **correccion automatica**: `ruff check --fix` aplica correcciones seguras sin intervencion manual.

Comandos principales:

```bash
# analizar sin modificar
ruff check .

# analizar y corregir automaticamente lo que sea seguro
ruff check --fix .

# verificar formato (equivalente a black --check)
ruff format --check .

# aplicar formato
ruff format .
```

## Configuracion de ruff.

ruff se configura en `pyproject.toml`, eliminando la necesidad de archivos separados como `.flake8` o `setup.cfg`:

```toml
[tool.ruff]
line-length = 88
target-version = "py310"

[tool.ruff.lint]
select = ["E", "W", "F", "I", "N", "UP", "S", "B"]
ignore = ["S101"]  # permite assert en tests

[tool.ruff.lint.per-file-ignores]
"tests/**/*.py" = ["S101", "S105"]

[tool.ruff.format]
quote-style = "double"
indent-style = "space"
```

La clave `select` define los grupos de reglas activos. La celda siguiente describe los grupos mas usados en proyectos Python de produccion.

In [ ]:
reglas_ruff = [
    ("E",   "pycodestyle errors",   "espaciado, longitud de linea, indentacion"),
    ("W",   "pycodestyle warnings", "espacios en blanco innecesarios"),
    ("F",   "pyflakes",             "variables e imports no usados"),
    ("I",   "isort",                "orden y agrupacion de imports"),
    ("N",   "pep8-naming",          "convencion de nombres de variables y clases"),
    ("UP",  "pyupgrade",            "actualizacion a sintaxis moderna de Python"),
    ("S",   "flake8-bandit",        "patrones de codigo con riesgo de seguridad"),
    ("B",   "flake8-bugbear",       "bugs potenciales y malas practicas"),
    ("C90", "mccabe",               "complejidad ciclomatica excesiva"),
]

print(f"{'Prefijo':<6} {'Origen':<25} Descripcion")
print("-" * 75)
for prefijo, origen, descripcion in reglas_ruff:
    print(f"{prefijo:<6} {origen:<25} {descripcion}")

## mypy: validacion de tipos estaticos.

`mypy` analiza las anotaciones de tipo del codigo Python sin ejecutarlo. Detecta errores como pasar un `str` donde se espera un `int`, retornar `None` cuando la firma declara un valor, o llamar a un atributo inexistente en un objeto.

En el pipeline, mypy actua como quality gate de correctitud logica: atrapa una clase de bugs que los tests no siempre cubren, porque los tests solo verifican los caminos que el desarrollador anticipo.

Configuracion recomendada en `pyproject.toml`:

```toml
[tool.mypy]
python_version = "3.10"
strict = false
ignore_missing_imports = true
warn_return_any = true
warn_unused_ignores = true
```

El modo `strict` activa todas las comprobaciones, incluyendo que cada funcion tenga anotaciones completas. Para proyectos nuevos es recomendable activarlo desde el inicio; para proyectos existentes conviene habilitarlo modulo por modulo.

La celda siguiente muestra como mypy detectaria un error de tipo antes de ejecutar el codigo.

In [ ]:
from typing import Optional

def procesar_usuario(nombre: str, edad: int, email: Optional[str] = None) -> dict:
    return {"nombre": nombre, "edad": edad, "email": email}

# mypy detectaria este error en tiempo de analisis, antes de ejecutar:
# procesar_usuario("Ana", "treinta")
# => Argument 2 to "procesar_usuario" has incompatible type "str"; expected "int"

resultado = procesar_usuario("Ana", 30, "ana@ejemplo.com")
print(resultado)

# Type alias para mayor legibilidad (sintaxis moderna Python 3.10+)
type UsuarioDict = dict[str, str | int | None]

def crear_respuesta(usuario: UsuarioDict, codigo: int = 200) -> dict[str, object]:
    return {"status": codigo, "data": usuario}

print(crear_respuesta(resultado))

## uv: dependencias deterministas.

`uv` es un gestor de paquetes y entornos para Python que reemplaza la combinacion `pip + venv + pip-compile`. Esta escrito en Rust y es entre 10 y 100 veces mas rapido que `pip`.

Su caracteristica central para DevSecOps es el **lockfile** (`uv.lock`): un archivo que fija las versiones exactas de todas las dependencias transitivas y sus hashes. Esto garantiza que el entorno del desarrollador, el CI y el contenedor de produccion sean identicos.

Sin lockfile, `pip install -r requirements.txt` puede instalar versiones distintas en diferentes momentos si una dependencia publica una nueva version. Con `uv.lock`, eso no puede ocurrir: la instalacion es reproducible y verificable.

```text
pyproject.toml     <- dependencias con rangos de version (ej. requests>=2.31)
uv.lock            <- versiones exactas y hashes de todo el arbol de dependencias
entorno virtual    <- instalacion real a partir del lockfile
```

El `uv.lock` debe versionarse en git. Si se modifica `pyproject.toml` sin actualizar el lockfile, `uv sync --frozen` falla en CI.

## Comandos esenciales de uv.

```bash
# crear un proyecto nuevo con pyproject.toml
uv init nombre-proyecto

# agregar una dependencia de produccion
uv add requests apiflask

# agregar dependencias de desarrollo
uv add --dev pytest ruff mypy pytest-cov

# instalar exactamente lo que dice uv.lock (modo CI)
uv sync --frozen

# ejecutar un comando dentro del entorno del proyecto
uv run pytest
uv run ruff check .
uv run mypy .

# actualizar dependencias y regenerar lockfile
uv lock --upgrade
```

La diferencia entre `uv sync` y `uv sync --frozen` es critica para CI: el primero puede modificar el lockfile si detecta diferencias; el segundo falla inmediatamente si el lockfile no coincide con `pyproject.toml`. En el pipeline siempre se usa `--frozen` para garantizar que nadie pueda introducir dependencias sin versionarlas.

## pipx: herramientas aisladas del proyecto.

`pipx` instala herramientas de linea de comandos en sus propios entornos virtuales, sin contaminar las dependencias del proyecto ni el Python del sistema.

Es el gestor recomendado para herramientas globales de desarrollo que no forman parte del proyecto como tal: `pre-commit`, `ruff` standalone, `uv`, o cualquier CLI que se use en varias repos a la vez:

```bash
# instalar una herramienta globalmente de forma aislada
pipx install ruff

# ejecutar una herramienta sin instalarla permanentemente
pipx run ruff check .

# listar herramientas instaladas con sus versiones
pipx list

# actualizar todas las herramientas instaladas
pipx upgrade-all
```

En el pipeline de CI, `uv` gestiona el entorno del proyecto. `pipx` es util en la maquina del desarrollador para mantener herramientas como `pre-commit` fuera del `pyproject.toml` y disponibles en todos los proyectos.

## Integracion en GitHub Actions.

El job de quality gates instala las dependencias con `uv` y ejecuta `ruff`, `mypy` y `pytest` como pasos secuenciales. Si cualquiera falla, el workflow se detiene y el merge queda bloqueado:

```yaml
jobs:
  quality:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - name: Instalar uv
        uses: astral-sh/setup-uv@v5

      - name: Instalar dependencias
        run: uv sync --frozen

      - name: Lint con ruff
        run: uv run ruff check .

      - name: Formato con ruff
        run: uv run ruff format --check .

      - name: Tipado con mypy
        run: uv run mypy .

      - name: Pruebas con pytest
        run: uv run pytest
```

El flag `--frozen` exige que `uv.lock` exista y coincida exactamente con `pyproject.toml`. Si alguien agrega una dependencia sin actualizar el lockfile, el pipeline falla antes de ejecutar una sola prueba.

In [ ]:
quality_gates = ['ruff check .', 'mypy .', 'pytest']

for indice, gate in enumerate(quality_gates, start=1):
    print(f'{indice}. {gate}')

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>